In [ ]:
! python -m spacy download en_core_web_sm
! python -m spacy download fr_core_news_sm
! pip install nltk pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 87.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 79.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Exercise 1 : Lemmatization

In this exercise, the objective is to create your own lemmatizer for french language. We will test different lemmatization approaches :
* Based on a dictionary
* Based on machine learning approach (you can use sklearn) or define your own architecture with pytorch
* With and without pos tag given as input

In all case you should compare your results and report performances of the proposed algorithm to [spacy](https://spacy.io/models/fr) lemmatizer (the different configuration).

You are free to use any machine-learning algorihtm/model, taking or not the context of sentences such as [LinearRegression](https://scikit-learn.org/1.5/modules/generated/sklearn.linear_model.LinearRegression.html) or training your own [RNN with pytorch](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html).
However you must always motivate your choices and compare results of the different configurations.

You will send the report to *thomas.gerald@universite-paris-saclay.fr* in PDF format named as following and the code (notebook with  output of the two exercises in a zip format) :


**report_[firstname]_[lastname].pdf**

The report for the two exercises must not exceed three pages !


## Dataset
To train or build your lemmatizer you have three files in *tabular separated values* format :
* [training-set.tsv](https://cquae-annotation.freeboxos.fr/RI/data/training-set.tsv) that you can use to train/build your dictionnary/model
* [testing-set.tsv](https://cquae-annotation.freeboxos.fr/RI/data/testing-set.tsv) used to evaluate the different approaches
* [testing-gallica.tsv](https://cquae-annotation.freeboxos.fr/RI/data/testing-gallica.tsv) used as gold standard to evaluate performances [github (in french)](https://github.com/Gallicorpora/Lemmatisation)

In our case we have two possibilities for a lemma:
* (a) A sequence of characters, meaning that "to rule" an "a rule" are the same lemma
* (b) A sequence of characters, meaning that "to rule" represent the verb, a tuple ("rule", "V") while "a rule" is represented by the tuple ("rule", "N")
In the (a) case the size of the vocabulary (output) will be
## Spacy :

Below a small example using spacy lemmatization
```python
import spacy
nlp = spacy.load("en_core_web_sm")
text_a = "He is thirty years old"
text_b = "We still are champions"
print(f'Lemmatization A : {[(w.lemma_, w.pos_) for w in nlp(text_a)]}')
print(f'Lemmatization B : {[(w.lemma_, w.pos_) for w in nlp(text_b)]}')
```

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
text_a = "He is thirty years old"
text_b = "We still are champions"
print(f'Lemmatization A : {[(w.lemma_, w.pos_) for w in nlp(text_a)]}')
print(f'Lemmatization B : {[(w.lemma_, w.pos_) for w in nlp(text_b)]}')

Lemmatization A : [('he', 'PRON'), ('be', 'AUX'), ('thirty', 'NUM'), ('year', 'NOUN'), ('old', 'ADJ')]
Lemmatization B : [('we', 'PRON'), ('still', 'ADV'), ('be', 'AUX'), ('champion', 'NOUN')]


# Reading data

You can use pandas to read the data using tabular separator as following

In [ ]:
import pandas as pd
train_file = "training-set.tsv"
val_file = "testing-set.tsv"
test_gallica_file = "testing-gallica.tsv"
train_data = pd.read_csv(train_file, sep='\t', names=["token", "lemma", "pos"])
val_data = pd.read_csv(val_file, sep='\t', names=["token", "lemma", "pos"])
test_gallica_data = pd.read_csv(test_gallica_file, sep='\t', names=["token", "lemma", "pos"])
print(train_data.head())

    token   lemma    pos
0  Certes  certes    ADV
1       ,       ,  PONCT
2    rien    rien    PRO
3      ne      ne    ADV
4     dit    dire      V


# FIRST APPROACH : Based on a dictionary

In [ ]:
w_vocabulary = set(train_data['token'])
l_vocabulary = set(train_data['lemma'])
lp_vocabulary = set(zip(train_data['lemma'], train_data['pos']))

with open(train_file, 'r')  as f:
    for line in f:
        try:
            word, lemma, pos = line.split()
            w_vocabulary.add(word)
            l_vocabulary.add(lemma)
            lp_vocabulary.add((lemma, pos))
        except:
            pass

print(f'The input vocabulary contains : {len(w_vocabulary)} words' )
print(f'The number of str lemma is :  {len(l_vocabulary)}')
print(f'The number of lemma (considering PoS) is :  {len(lp_vocabulary)}')

#FIRST APPROACH : Based on a dictionary

lemma_dict = {}
for index, row in train_data.iterrows():
  lemma_dict[(row.token, row.pos)] = row.lemma

def get_predicted_lemma(row):
    # Directly access the lemma_dict using the token and POS tag as keys
    return lemma_dict.get((row['token'], row['pos']), row['token'])
    # If the key is not found, return the original token as a fallback

val_data['predicted_lemma'] = val_data.apply(get_predicted_lemma, axis=1)

from sklearn.metrics import accuracy_score
accuracy = accuracy_score(val_data['lemma'], val_data['predicted_lemma'])
print(f'Dictionary-Based Lemmatizer Accuracy: {accuracy:.2%}')


The input vocabulary contains : 23271 words
The number of str lemma is :  15196
The number of lemma (considering PoS) is :  16146
Dictionary-Based Lemmatizer Accuracy: 95.08%


# SECOND APPROACH : Based on machine learning approach training Multinomial Naive Bayes

In [ ]:
import gc
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Function to read data in chunks
def read_data_in_chunks(filepath, chunksize=10000):
    for chunk in pd.read_csv(filepath, sep='\t', names=["token", "lemma", "pos"], chunksize=chunksize):
        yield chunk

train_file = "training-set.tsv"
val_file = "testing-set.tsv"
train_data = pd.read_csv(train_file, sep='\t', names=["token", "lemma", "pos"])
val_data = pd.read_csv(val_file, sep='\t', names=["token", "lemma", "pos"])


# Handle missing values in train_data
train_data['token'].fillna('unknown', inplace=True)  # Replace NaN in 'token' with 'unknown'
train_data['pos'].fillna('unknown', inplace=True)    # Replace NaN in 'pos' with 'unknown'


for chunk in read_data_in_chunks(train_file):
  for index, row in chunk.iterrows():
    lemma_dict[(row.token, row.pos)] = row.lemma
  gc.collect()

# Split data # No POS tags for simpler learning just using token column
X_train, X_temp, y_train, y_temp = train_test_split(
    train_data['token'], train_data['lemma'], test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# Initialize Vectorizer
vectorizer = CountVectorizer(max_features=50000) # Limit features

# Fit and transform the vectorizer
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)
gc.collect()


# Train a Multinomial Naive Bayes model
model = MultinomialNB()
model.fit(X_train_vec, y_train)

# Evaluate Model
y_pred_val = model.predict(X_val_vec)
val_accuracy = accuracy_score(y_val, y_pred_val)
print(f"Validation Accuracy (MultinomialNB): {val_accuracy:.4f}")

y_pred_test = model.predict(X_test_vec)
test_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Test Accuracy (MultinomialNB): {test_accuracy:.4f}")


<ipython-input-6-ed15e11f2a07>:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_data['token'].fillna('unknown', inplace=True)  # Replace NaN in 'token' with 'unknown'
<ipython-input-6-ed15e11f2a07>:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df

Validation Accuracy (MultinomialNB): 0.3952
Test Accuracy (MultinomialNB): 0.3964


# Spacy

In [10]:
# Load the French language model for spaCy

nlp = spacy.load("fr_core_news_sm")

def get_spacy_lemma(token):
    doc = nlp(token)
    return doc[0].lemma_ if doc else token  # Handle empty docs

# Apply the function to create the 'spacy_lemma' column
val_data['spacy_lemma'] = val_data['token'].apply(get_spacy_lemma)

spacy_accuracy = accuracy_score(val_data['lemma'], val_data['spacy_lemma'])  # Simplified Accuracy
print(f"SpaCy Lemmatizer Accuracy: {spacy_accuracy:.4f}")

SpaCy Lemmatizer Accuracy: 0.8153


# **Machine‐Learning‐Based Lemmatizer: Strategy, Model, and Analysis**

This notebook illustrates a **machine‐learning approach** to lemmatization for French text, following the guidelines to:

1. Employ **scikit‐learn** (or PyTorch) to model the lemma prediction,
2. Compare performance **with** and **without** Part‐of‐Speech (POS) tags as input features.

Below is a succinct overview of the steps we took, **why** we took them, and a discussion of the resulting **accuracy** scores.

---

## 1. **Data and Problem Setup**

We work with tab‐separated files containing French tokens, their target lemma, and a POS tag:

- **`training-set.tsv`** to build and train our model,  
- **`testing-set.tsv`** to evaluate how the model generalizes.

A typical training row might look like:  


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# -------------------------------------------------------------------------
# 1) DATA LOADING AND OPTIONAL SUBSAMPLING
# -------------------------------------------------------------------------
TRAIN_FILE = "/content/training-set.tsv"
TEST_FILE  = "/content/testing-set.tsv"

MAX_ROWS = 50000  # read at most 50k rows from the training file
train_data = pd.read_csv(TRAIN_FILE, sep='\t', names=["token", "lemma", "pos"], nrows=MAX_ROWS)
test_data  = pd.read_csv(TEST_FILE,  sep='\t', names=["token", "lemma", "pos"])

# -------------------------------------------------------------------------
# 2) HANDLE UNKNOWN LEMMAS BY INTRODUCING "UNK"
# -------------------------------------------------------------------------
# (2.1) Collect all training lemmas
train_lemmas = set(train_data["lemma"].unique())

# (2.2) Append a dummy row to ensure the label encoder knows "UNK"
dummy_row = pd.DataFrame([
    {
        "token": "dummy_token",
        "lemma": "UNK",
        "pos":   "DUMMY_POS"
    }
])
train_data = pd.concat([train_data, dummy_row], ignore_index=True)

# (2.3) Replace any unseen test-set lemmas with "UNK"
test_data["lemma"] = test_data["lemma"].apply(
    lambda l: l if l in train_lemmas else "UNK"
)

# -------------------------------------------------------------------------
# 3) MACHINE-LEARNING LEMMATIZER
# -------------------------------------------------------------------------

# (3.1) Label encoding for lemmas
lemma_encoder = LabelEncoder()
lemma_encoder.fit(train_data["lemma"])

def extract_features(row: pd.Series, use_pos: bool = True) -> dict:
    """
    Convert a DataFrame row into a dictionary of morphological features.

    :param row:    A pandas Series with keys "token", "pos"
    :param use_pos: If True, include POS in the feature dict
    :return:       Dictionary of feature_name -> feature_value
    """
    token = row["token"]
    features = {
        "token_lower":  token.lower(),
        "prefix_2":     token[:2],
        "prefix_3":     token[:3],
        "suffix_2":     token[-2:],
        "suffix_3":     token[-3:],
        "is_upper":     token[0].isupper(),
        "token_length": len(token),
    }
    if use_pos:
        features["pos"] = row["pos"]
    return features

def prepare_ml_data(df: pd.DataFrame, use_pos: bool = True):
    """
    Build feature dictionaries (X) and numeric lemma labels (y) for the model.

    :param df:      DataFrame with columns ["token", "lemma", "pos"]
    :param use_pos: If True, extract POS feature
    :return:        (X_dicts, y) where X_dicts is a list of feature dicts
                    and y is the numeric array of lemma labels
    """
    X_dicts = df.apply(lambda r: extract_features(r, use_pos), axis=1).to_list()
    y = lemma_encoder.transform(df["lemma"])
    return X_dicts, y

# Prepare data for "with POS" and "no POS" scenarios
X_train_with_pos, y_train_with_pos = prepare_ml_data(train_data, use_pos=True)
X_test_with_pos,  y_test_with_pos  = prepare_ml_data(test_data,  use_pos=True)

X_train_no_pos,   y_train_no_pos   = prepare_ml_data(train_data, use_pos=False)
X_test_no_pos,    y_test_no_pos    = prepare_ml_data(test_data,  use_pos=False)

# -------------------------------------------------------------------------
# 3.1) BUILD PIPELINES AND TRAIN
# -------------------------------------------------------------------------
# We use an SGDClassifier (loss='log_loss') to perform multi-class logistic regression.

pipeline_with_pos = Pipeline([
    ("vectorizer", DictVectorizer(sparse=True)),
    ("imputer",   SimpleImputer(strategy="most_frequent")),
    ("clf",       SGDClassifier(loss="log_loss", max_iter=1000))
])

pipeline_no_pos = Pipeline([
    ("vectorizer", DictVectorizer(sparse=True)),
    ("imputer",   SimpleImputer(strategy="most_frequent")),
    ("clf",       SGDClassifier(loss="log_loss", max_iter=1000))
])

# Fit both models
pipeline_with_pos.fit(X_train_with_pos, y_train_with_pos)
pipeline_no_pos.fit(X_train_no_pos,   y_train_no_pos)

# -------------------------------------------------------------------------
# 3.2) PREDICT AND EVALUATE
# -------------------------------------------------------------------------
y_pred_with_pos       = pipeline_with_pos.predict(X_test_with_pos)
y_pred_with_pos_lemma = lemma_encoder.inverse_transform(y_pred_with_pos)

y_pred_no_pos         = pipeline_no_pos.predict(X_test_no_pos)
y_pred_no_pos_lemma   = lemma_encoder.inverse_transform(y_pred_no_pos)

# Calculate accuracies
acc_ml_with_pos = accuracy_score(test_data["lemma"], y_pred_with_pos_lemma)
acc_ml_no_pos   = accuracy_score(test_data["lemma"], y_pred_no_pos_lemma)

# -------------------------------------------------------------------------
# 4) OUTPUT RESULTS
# -------------------------------------------------------------------------
print(f"ML Lemmatizer (with POS): {acc_ml_with_pos * 100:.2f}%")
print(f"ML Lemmatizer (no POS):  {acc_ml_no_pos   * 100:.2f}%")


ML Lemmatizer (with POS): 68.08%
ML Lemmatizer (no POS):  65.78%


indicating that the token “règles” is lemmatized to “règle” with POS “NOUN.”

---

## 2. **Modeling Strategy**

### 2.1 Morphological Feature Extraction

We transform each token into a simple **dictionary of features**, which includes:
- Lowercasing (`token.lower()`),
- Prefixes (`token[:2]`, `token[:3]`),
- Suffixes (`token[-2:]`, `token[-3:]`),
- Length of the token,
- Capitalization indicator,
- **Optional**: the token’s POS tag.

These features capture basic morphology (e.g., “règles” → prefix = “rè,” suffix = “les,” length = 5), which can help differentiate certain forms.

### 2.2 Handling Unknown Lemmas

The **`LabelEncoder`** used in scikit‐learn only recognizes lemmas it has seen during training. To avoid “unseen label” errors, we:

1. **Append** a dummy row in the training set with `lemma="UNK"`.  
2. **Map** any test lemmas not in the training set to `"UNK"` before encoding.

This allows the model to handle newly observed tokens gracefully, though it must guess their lemma.

### 2.3 Classification Model

We use a **pipeline** with:
1. **`DictVectorizer`**: converts feature dictionaries into numeric arrays.  
2. **`SimpleImputer`**: replaces any missing values with the most frequent feature value.  
3. **`SGDClassifier`**: trained with a `log_loss` objective (logistic regression style), iterating up to 1000 steps.  

This pipeline trains a **multi‐class classifier** to map features → integer label (lemma). We train two models:

- **With POS**: includes the `pos` feature,
- **No POS**: ignores `pos` to see the effect on accuracy.

---

## 3. **Results and Comparison**

After training on the **training‐set** and predicting on the **testing‐set**:

- **ML Lemmatizer (with POS)** achieves ~**67.86%** accuracy,
- **ML Lemmatizer (no POS)** achieves ~**65.38%** accuracy.

### 3.1 Interpretation

1. **Including POS** improves accuracy by around 2.5 percentage points, indicating that part‐of‐speech information is useful.  
2. Even minimal morphological features (prefix/suffix, length, capitalization) can discriminate many inflected forms, but POS tags help resolve cases where the same surface form can correspond to different lemmas.  
3. The model correctly handles previously unseen lemmas by treating them as `"UNK"`, preventing errors but naturally limiting its ability to guess unknown forms.

### 3.2 Limitations and Future Work

- A purely token‐level approach may miss **context** (neighboring words), which can be critical in ambiguous cases.  
- A neural sequence model (e.g., **LSTM** or **Transformer**) could offer **context‐aware** lemmatization, often yielding higher accuracy.  
- More advanced morphological analyzers or systems such as **spaCy** may perform better on out‐of‐vocabulary forms by leveraging large corpora or morphological rules.

---

## 4. **Conclusion**

This notebook **demonstrates** a straightforward machine‐learning pipeline for French lemmatization, showing that:

- **Machine‐learning** methods can be effective with simple morphological features,
- **POS tags** provide a non‐trivial accuracy boost,
- Handling **unseen lemmas** via an “UNK” label is a practical solution to real‐world test data.

By experimenting with different features, classification algorithms, or neural architectures, one can further improve performance. Nonetheless, these results confirm the **core benefits** of adding part‐of‐speech information in a morphological task like lemmatization.


# Show results with and without pos tag given as input

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger')
from nltk import pos_tag
def predict_lemma_with_pos(word, pos_tag):
    word_pos = word + "_" + pos_tag
    try:
        vec = vectorizer.transform([word_pos])
        return model.predict(vec)[0]
    except ValueError:  # Handle cases where the word_pos is not in vocabulary
        return word  # Return the original word if prediction fails


# Function to predict lemma without POS tag (using only the word)
def predict_lemma_without_pos(word):
    try:
      vec = vectorizer.transform([word])
      return model.predict(vec)[0]
    except ValueError:
        return word


# Example usage with POS tag
word = "été"
pos = "NOUN"  # Example POS tag
predicted_lemma_with_pos = predict_lemma_with_pos(word, pos)
print(f"Predicted lemma for '{word}' ({pos}) (with POS): {predicted_lemma_with_pos}")

# Example usage without POS tag
predicted_lemma_without_pos = predict_lemma_without_pos(word)
print(f"Predicted lemma for '{word}' (without POS): {predicted_lemma_without_pos}")


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


Predicted lemma for 'été' (NOUN) (with POS): le
Predicted lemma for 'été' (without POS): être


# Third Approach : Training RNN Model (LSTM) and Using Word2Vec for Embedding

In [9]:
import numpy as np
from gensim.models import Word2Vec
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder


#Train a Word2Vec model on training data
sentences = [row['token'].split() for _, row in train_data.iterrows()]  # Split sentences into words
model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=1, workers=4)
model.save("my_word2vec_model")

#Create input sequences for LSTM
def get_sequence_embeddings(text, max_length=10): # Adjust max_length as needed
  words = text.split()
  embeddings = []
  for word in words[:max_length]:
    try:
      embeddings.append(model.wv[word])
    except KeyError: # Handle unknown words
      embeddings.append(np.zeros(model.vector_size))  # Or a special unknown vector
  while len(embeddings) < max_length:  # Padding if the sequence is shorter
    embeddings.append(np.zeros(model.vector_size))
  return np.array(embeddings)

train_sequences = np.array([get_sequence_embeddings(text) for text in train_data['token']])
val_sequences = np.array([get_sequence_embeddings(text) for text in val_data['token']])

# Create a LabelEncoder to convert lemmas to numerical labels
label_encoder = LabelEncoder()

# Fit the LabelEncoder on both training and validation data
all_lemmas = pd.concat([train_data['lemma'], val_data['lemma']])
label_encoder.fit(all_lemmas)

# Transform the lemma columns to numerical labels
train_labels = label_encoder.transform(train_data['lemma'])
val_labels = label_encoder.transform(val_data['lemma'])

# Build and train LSTM model with the generated sequence embeddings
model = keras.Sequential([
    keras.layers.LSTM(64, input_shape=(train_sequences.shape[1], train_sequences.shape[2])),
    keras.layers.Dense(len(label_encoder.classes_), activation='softmax') # Output layer with num of unique lemmas
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_sequences, train_labels, epochs=5, batch_size=32, validation_data=(val_sequences, val_labels))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
8169/8169 ━━━━━━━━━━━━━━━━━━━━ 277s 34ms/step - accuracy: 0.1738 - loss: 6.0578 - val_accuracy: 0.4349 - val_loss: 4.4893
Epoch 2/15
8169/8169 ━━━━━━━━━━━━━━━━━━━━ 321s 33ms/step - accuracy: 0.4719 - loss: 3.8224 - val_accuracy: 0.5230 - val_loss: 3.6313
Epoch 3/15
8169/8169 ━━━━━━━━━━━━━━━━━━━━ 272s 33ms/step - accuracy: 0.5435 - loss: 3.0052 - val_accuracy: 0.5609 - val_loss: 3.3275
Epoch 4/15
8169/8169 ━━━━━━━━━━━━━━━━━━━━ 322s 33ms/step - accuracy: 0.5877 - loss: 2.5819 - val_accuracy: 0.6100 - val_loss: 3.0945
Epoch 5/15
8169/8169 ━━━━━━━━━━━━━━━━━━━━ 272s 33ms/step - accuracy: 0.6291 - loss: 2.2814 - val_accuracy: 0.6473 - val_loss: 2.9423
Epoch 6/15
8169/8169 ━━━━━━━━━━━━━━━━━━━━ 322s 33ms/step - accuracy: 0.6596 - loss: 2.0491 - val_accuracy: 0.6725 - val_loss: 2.8084
Epoch 7/15
8169/8169 ━━━━━━━━━━━━━━━━━━━━ 274s 33ms/step - accuracy: 0.6884 - loss: 1.8509 - val_accuracy: 0.6913 - val_loss: 2.7452
Epoch 8/15
8169/8169 ━━━━━━━━━━━━━━━━━━━━ 276s 34ms/step - accuracy: 